# Statistik mit Python

**Einführung anhand von Messdaten eines Web-Systems**

In diesem Notebook werden grundlegende Verfahren der deskriptiven Statistik mit Python angewendet. Im Mittelpunkt stehen die statistischen Fragestellungen; Python dient zur Aufbereitung, Auswertung und grafischen Darstellung der Daten.

## Lernziele

Nach der Bearbeitung des Notebooks können Sie

- Grundgesamtheit, Merkmale und Ausprägungen eines Datensatzes benennen,
- nominalskalierte und metrisch skalierte Merkmale unterscheiden,
- Daten auswählen, filtern und gruppieren,
- absolute und relative Häufigkeiten bestimmen,
- arithmetisches Mittel, Median, Quartile, Interquartilsabstand und empirische Standardabweichung berechnen und interpretieren,
- Säulendiagramm, Histogramm, Boxplot und Streudiagramm sachgerecht einsetzen,
- den Pearson-Korrelationskoeffizienten interpretieren,
- deskriptive Aussagen von kausalen Aussagen unterscheiden und
- die zufällige Schwankung von Stichprobenmittelwerten erläutern.

Eine Code-Zelle wird mit **Shift + Enter** ausgeführt.

## Fragestellungen

Wir betrachten Messdaten eines fiktiven Web-Systems. Jede Zeile des Datensatzes beschreibt einen API-Request.

1. Welche Endpoints werden häufig aufgerufen?
2. Wie unterscheiden sich die Antwortzeiten zwischen den Endpoints?
3. Wie sind die Antwortzeiten verteilt und wie stark streuen sie?
4. Besteht ein linearer Zusammenhang zwischen Payload-Größe und Antwortzeit?
5. Unterscheiden sich die Antwortzeiten bei Requests mit und ohne Cache-Hit?
6. Wie stark schwankt das arithmetische Mittel zwischen verschiedenen Zufallsstichproben?

## 1. Untersuchungsgegenstand und Merkmale

Ein **API-Request** ist eine Anfrage eines Programms an einen Server. Ein **Endpoint** bezeichnet die angesprochene Funktion der Schnittstelle, beispielsweise `/search` oder `/checkout`. Die **Payload** ist die übertragene Datenmenge. Ein **Cache** ist ein Zwischenspeicher; bei einem **Cache-Hit** können bereits gespeicherte Daten verwendet werden.

Die Variablen im Datensatz haben folgende Bedeutung:

| Variable | Bedeutung |
|---|---|
| `request_id` | Kennung des Requests |
| `endpoint` | angesprochener API-Endpoint |
| `region` | Region des Requests |
| `hour` | Stunde des Tages |
| `cache_hit` | gibt an, ob Daten aus dem Cache verwendet wurden |
| `payload_kb` | Payload-Größe in Kilobyte |
| `response_ms` | Antwortzeit in Millisekunden |
| `status_code` | HTTP-Statuscode, z. B. 200, 404 oder 500 |

### Grundbegriffe

- **Grundgesamtheit:** die Gesamtheit der für die Untersuchung interessanten Daten liefernden Objekte; hier können das die interessierenden API-Requests sein.
- **Merkmal:** eine für die Untersuchung interessante Eigenschaft eines solchen Objekts, z. B. Antwortzeit oder Endpoint.
- **Ausprägung:** ein beobachteter Wert eines Merkmals.
- **Wertemenge:** die Menge aller möglichen Ausprägungen eines Merkmals.
- **Stichprobe:** die tatsächlich untersuchte Teilmenge der Grundgesamtheit.

Die 320 Zeilen bilden zunächst die vorliegenden Beobachtungsdaten. Ob diese Daten als Stichprobe einer größeren Grundgesamtheit aufgefasst werden können, hängt von der Fragestellung und der Art der Datenerhebung ab.

### Merkmalsskalierungen

- `endpoint`, `region`, `cache_hit` und `status_code` sind **nominalskaliert**.
- `payload_kb` und `response_ms` sind **metrisch skaliert**.
- `hour` ist eine diskrete Zeitangabe und wird im Folgenden nicht für Lage- oder Streuungsmaße verwendet.

Dass `status_code` durch Zahlen dargestellt wird, macht das Merkmal nicht metrisch. Differenzen oder Mittelwerte von HTTP-Statuscodes haben keine sinnvolle inhaltliche Interpretation.

## 2. Benötigte Python-Bibliotheken

Wir verwenden `pandas` zur Arbeit mit Tabellen, `numpy` für numerische Berechnungen und Zufallszahlen sowie `matplotlib` für grafische Darstellungen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (8, 4.5)

## 3. Datensatz einlesen und prüfen

In [ ]:
df = pd.read_csv("data/api_requests.csv")
df.head()

In [ ]:
print("Anzahl Zeilen und Spalten:", df.shape)
print("Spalten:", df.columns.to_list())

In [ ]:
df.info()

Vor einer statistischen Auswertung sollte geprüft werden, ob Werte fehlen. Fehlende Werte werden nicht pauschal entfernt. Für jede Auswertung ist zu entscheiden, welche Variablen benötigt werden.

In [ ]:
df.isna().sum()

Für Auswertungen der Antwortzeit benötigen wir Beobachtungen mit vorhandenem Wert von `response_ms`. Dafür wird ein eigener DataFrame angelegt. Bei Auswertungen anderer Merkmale kann weiterhin der vollständige Datensatz verwendet werden.

In einer realen Untersuchung müsste zusätzlich geklärt werden, weshalb Werte fehlen. Das Entfernen unvollständiger Beobachtungen kann zu Verzerrungen führen, wenn das Auftreten fehlender Werte systematisch ist.

In [ ]:
response_data = df.dropna(subset=["response_ms"]).copy()

print("Alle Requests:", len(df))
print("Requests mit gemessener Antwortzeit:", len(response_data))

## 4. Daten auswählen und filtern

Mit Bedingungen lassen sich Teilmengen eines DataFrames auswählen.

In [ ]:
# Eine Spalte auswählen
response_data["response_ms"].head()

In [ ]:
# Requests mit HTTP-Statuscode 500
server_errors = df[df["status_code"] == 500]
server_errors[["request_id", "endpoint", "response_ms", "status_code"]].head()

In [ ]:
# Mehrere Bedingungen verknüpfen
slow_checkout = response_data[
    (response_data["endpoint"] == "/checkout")
    & (response_data["response_ms"] > 400)
]

slow_checkout.head()

### Übung 1: Filtern

Filtern Sie alle Requests, die

- den Endpoint `/search` verwenden,
- aus `EU-Central` stammen und
- eine Antwortzeit von mehr als 250 ms haben.

Bestimmen Sie die Anzahl dieser Requests.

Überlegen Sie anschließend: Welche Aussage beschreibt nur die vorliegenden Beobachtungen, und welche Aussage wäre bereits eine Verallgemeinerung auf eine größere Grundgesamtheit?

In [ ]:
# TODO: Filter formulieren und Anzahl bestimmen.

## 5. Nominalskalierte Merkmale: absolute und relative Häufigkeiten

Für die Frage nach der Nutzung der Endpoints werden alle Requests betrachtet. Fehlende Antwortzeiten spielen für diese Auswertung keine Rolle.

In [ ]:
# Absolute Häufigkeiten
endpoint_counts = df["endpoint"].value_counts()
endpoint_counts

In [ ]:
# Relative Häufigkeiten in Prozent
endpoint_percent = df["endpoint"].value_counts(normalize=True) * 100
endpoint_percent.round(1)

In [ ]:
endpoint_counts.plot(kind="bar")
plt.title("Absolute Häufigkeiten der Endpoints")
plt.xlabel("Endpoint")
plt.ylabel("Absolute Häufigkeit")
plt.xticks(rotation=0)
plt.show()

Für nominalskalierte Merkmale sind Häufigkeitstabellen und Säulendiagramme geeignete Darstellungen. Ein Histogramm wird dagegen zur Darstellung der Verteilung eines metrisch skalierten Merkmals verwendet.

## 6. Metrisch skalierte Merkmale: Lage und Streuung

Für die Antwortzeit betrachten wir verschiedene Lage- und Streuungsmaße.

In [ ]:
response_data["response_ms"].describe()

In [ ]:
x = response_data["response_ms"]

q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1

print("Arithmetisches Mittel:       ", round(x.mean(), 1), "ms")
print("Median:                      ", round(x.median(), 1), "ms")
print("empirische Standardabweichung:", round(x.std(), 1), "ms")
print("unteres Quartil:             ", round(q1, 1), "ms")
print("oberes Quartil:             ", round(q3, 1), "ms")
print("Interquartilsabstand:        ", round(iqr, 1), "ms")

### Arithmetisches Mittel und Median

Das **arithmetische Mittel** berücksichtigt alle beobachteten Werte und reagiert deshalb empfindlich auf sehr große oder sehr kleine Beobachtungen. Der **Median** teilt die geordneten Beobachtungen so, dass mindestens 50 % der Werte kleiner oder gleich und mindestens 50 % größer oder gleich dem Median sind. Er ist gegenüber einzelnen Extremwerten robuster.

In [ ]:
response_data.nlargest(8, "response_ms")[[
    "request_id", "endpoint", "region", "response_ms", "status_code"
]]

In [ ]:
response_data["response_ms"].plot.hist(bins=28, edgecolor="black")
plt.axvline(response_data["response_ms"].mean(), linestyle="--", label="Arithmetisches Mittel")
plt.axvline(response_data["response_ms"].median(), linestyle=":", label="Median")
plt.title("Histogramm der Antwortzeiten")
plt.xlabel("Antwortzeit [ms]")
plt.ylabel("Absolute Häufigkeit")
plt.legend()
plt.show()

Die Verteilung ist rechtsschief. Einzelne große Antwortzeiten erhöhen das arithmetische Mittel stärker als den Median. Bei schiefen Verteilungen ist es daher sinnvoll, mehrere Lage- und Streuungsmaße gemeinsam zu betrachten.

Auch die Wahl der Klassenbreite eines Histogramms beeinflusst den visuellen Eindruck der Verteilung. Ein Histogramm sollte deshalb nicht isoliert interpretiert werden.

In [ ]:
response_data[["response_ms"]].boxplot()
plt.title("Boxplot der Antwortzeiten")
plt.ylabel("Antwortzeit [ms]")
plt.show()

### Boxplot

Der Boxplot stellt insbesondere Median, Quartile und Interquartilsabstand dar.

- Die Linie innerhalb der Box kennzeichnet den Median.
- Die untere und obere Boxkante entsprechen dem unteren und oberen Quartil.
- Die Länge der Box entspricht dem Interquartilsabstand $IQR = Q_3-Q_1$.
- Standardmäßig werden Beobachtungen außerhalb der Grenzen $Q_1-1{,}5\cdot IQR$ und $Q_3+1{,}5\cdot IQR$ als einzelne Punkte dargestellt. Die Whisker reichen bis zu den äußersten beobachteten Werten innerhalb dieser Grenzen.

Solche Beobachtungen werden häufig als Ausreißer bezeichnet. Sie sind nicht automatisch fehlerhafte Messwerte.

## 7. Gruppenvergleich: Endpoint und Antwortzeit

Nun wird ein nominalskaliertes Merkmal (`endpoint`) gemeinsam mit einem metrisch skalierten Merkmal (`response_ms`) betrachtet.

In [ ]:
endpoint_stats = (
    response_data.groupby("endpoint")["response_ms"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("median")
    .round(1)
)

endpoint_stats

In der Tabelle stehen `mean` für das arithmetische Mittel und `std` für die empirische Standardabweichung.

In [ ]:
response_data.boxplot(column="response_ms", by="endpoint", grid=False)
plt.suptitle("")
plt.title("Antwortzeiten nach Endpoint")
plt.xlabel("Endpoint")
plt.ylabel("Antwortzeit [ms]")
plt.xticks(rotation=20)
plt.show()

### Fragen zur Auswertung

- Welcher Endpoint hat den größten Median der Antwortzeit?
- Bei welchem Endpoint ist die Streuung besonders groß?
- Bei welchen Endpoints treten Ausreißer auf?
- Unterscheiden sich arithmetisches Mittel und Median innerhalb der Gruppen deutlich?

Die Gruppenunterschiede sind zunächst deskriptive Befunde. Aus ihnen folgt noch keine Aussage über die Ursachen der Unterschiede.

## 8. Antwortzeit und Cache-Hit

Untersuchen Sie, ob sich die beobachteten Antwortzeiten zwischen Requests mit und ohne Cache-Hit unterscheiden.

Bestimmen Sie für beide Gruppen

- die Anzahl der Beobachtungen,
- das arithmetische Mittel und
- den Median der Antwortzeit.

In [ ]:
# TODO: Nach "cache_hit" gruppieren und "response_ms" auswerten.

### Diskussion

Ein Unterschied zwischen den beiden Gruppen zeigt einen statistischen Zusammenhang in den vorliegenden Daten. Daraus folgt nicht unmittelbar, dass der Cache-Hit die gesamte beobachtete Differenz verursacht.

Mögliche Drittvariablen sind beispielsweise Endpoint, Region, Payload-Größe oder Tageszeit. Prüfen Sie zunächst, ob Cache-Hits bei allen Endpoints gleich häufig auftreten.

In [ ]:
pd.crosstab(df["endpoint"], df["cache_hit"], normalize="index").round(2)

Die relativen Häufigkeiten unterscheiden sich zwischen den Endpoints. Der einfache Vergleich der beiden Cache-Gruppen ist daher eine deskriptive Auswertung, aber kein Nachweis einer kausalen Wirkung.

## 9. Zwei metrisch skalierte Merkmale: Streudiagramm und Korrelation

Wir untersuchen den Zusammenhang zwischen Payload-Größe und Antwortzeit. Zunächst wird ein Streudiagramm betrachtet.

In [ ]:
response_data.plot.scatter(x="payload_kb", y="response_ms", alpha=0.6)
plt.title("Payload-Größe und Antwortzeit")
plt.xlabel("Payload-Größe [kB]")
plt.ylabel("Antwortzeit [ms]")
plt.show()

In [ ]:
r = response_data["payload_kb"].corr(response_data["response_ms"])
print("Pearson-Korrelationskoeffizient r_x,y:", round(r, 3))

Der **Pearson-Korrelationskoeffizient** $r_{x,y}$ beschreibt Richtung und Stärke der linearen Korrelation zweier metrisch skalierter Merkmale. Es gilt $-1\le r_{x,y}\le 1$.

- $r_{x,y}=1$: perfekter positiver linearer Zusammenhang,
- $r_{x,y}=-1$: perfekter negativer linearer Zusammenhang,
- $r_{x,y}=0$: die beiden Merkmale sind linear unkorreliert.

Für die Einteilung in „schwach“, „mittel“ oder „stark“ gibt es keine allgemein gültigen Grenzwerte; die Beurteilung hängt vom Anwendungskontext ab.

Ein Wert nahe null schließt einen nichtlinearen Zusammenhang oder unterschiedliche Zusammenhänge in Teilgruppen nicht aus. Der Pearson-Korrelationskoeffizient kann außerdem empfindlich auf Ausreißer reagieren.

Aus einer Korrelation folgt keine Kausalität.

Im vorliegenden Datensatz ist der lineare Gesamtzusammenhang zwischen Payload-Größe und Antwortzeit gering. Das Ergebnis widerspricht nicht der Möglichkeit, dass innerhalb einzelner Teilgruppen andere Zusammenhänge bestehen oder weitere Merkmale die Antwortzeit beeinflussen.

## 10. Zufällige Stichproben und Stichprobenmittelwerte

Für die folgende Simulation werden die Requests mit gemessener Antwortzeit als endliche Grundgesamtheit aufgefasst. Aus dieser Grundgesamtheit werden wiederholt einfache Zufallsstichproben ohne Zurücklegen gezogen.

Für jede Stichprobe mit Umfang $n=30$ wird das arithmetische Mittel der Antwortzeiten berechnet.

In [ ]:
rng = np.random.default_rng(42)

sample_means = []
for _ in range(500):
    sample = response_data["response_ms"].sample(
        n=30,
        replace=False,
        random_state=int(rng.integers(1_000_000))
    )
    sample_means.append(sample.mean())

sample_means = pd.Series(sample_means)
sample_means.describe()

In [ ]:
sample_means.plot.hist(bins=25, edgecolor="black")
plt.axvline(
    response_data["response_ms"].mean(),
    linestyle="--",
    label="Arithmetisches Mittel der Grundgesamtheit"
)
plt.title("Verteilung von 500 Stichprobenmittelwerten (n = 30)")
plt.xlabel("Stichprobenmittelwert der Antwortzeit [ms]")
plt.ylabel("Absolute Häufigkeit")
plt.legend()
plt.show()

Die Stichprobenmittelwerte sind nicht identisch, obwohl alle Stichproben aus derselben Grundgesamtheit gezogen werden. Die Stichprobenmittelwerte unterliegen also einer **zufälligen Schwankung**.

In [ ]:
def repeated_means(n, repetitions=500, seed=123):
    local_rng = np.random.default_rng(seed)
    means = []
    for _ in range(repetitions):
        sample = response_data["response_ms"].sample(
            n=n,
            replace=False,
            random_state=int(local_rng.integers(1_000_000))
        )
        means.append(sample.mean())
    return pd.Series(means)

comparison = pd.DataFrame({
    "n=10": repeated_means(10),
    "n=30": repeated_means(30),
    "n=100": repeated_means(100),
})

comparison.std().round(1)

In dieser Simulation nimmt die Standardabweichung der Stichprobenmittelwerte mit zunehmendem Stichprobenumfang ab. Größere Zufallsstichproben liefern daher im Allgemeinen präzisere Schätzungen eines Mittelwerts, sofern das verwendete Stichprobenverfahren und die zugrunde liegenden Annahmen angemessen sind.

## 11. Abschlussaufgaben

Beantworten Sie die folgenden Fragen zunächst mit Python und formulieren Sie anschließend jeweils eine kurze statistische Aussage.

1. Welcher Endpoint hat den größten Median der Antwortzeit?
2. Wie groß ist die relative Häufigkeit der Requests mit Statuscode 500?
3. Welche Region hat den größten Median der Antwortzeit?
4. Formulieren Sie zu Aufgabe 3 eine rein deskriptive Aussage, ohne eine Ursache für den Unterschied zu behaupten.

In [ ]:
# Aufgabe 1

In [ ]:
# Aufgabe 2

In [ ]:
# Aufgabe 3

### Beurteilung statistischer Aussagen

Welche der folgenden Aussagen sind durch die bisherigen Auswertungen gerechtfertigt?

A. Im beobachteten Datensatz hat `/checkout` den größten Median der Antwortzeit.

B. Aus dem Unterschied zwischen Requests mit und ohne Cache-Hit lässt sich unmittelbar bestimmen, um wie viele Millisekunden ein Cache-Hit die Antwortzeit kausal verkürzt.

C. Ein Pearson-Korrelationskoeffizient nahe null beweist, dass zwischen Payload-Größe und Antwortzeit keinerlei Zusammenhang bestehen kann.

## 12. Zusammenfassung

- Statistische Auswertungen beginnen mit der Festlegung von Grundgesamtheit, Merkmalen und Ausprägungen.
- Die Merkmalsskalierung bestimmt, welche Kennzahlen und Darstellungen sinnvoll sind.
- Fehlende Werte sollten in Abhängigkeit von der jeweiligen Fragestellung behandelt werden.
- Arithmetisches Mittel und Median sind Lagemaße mit unterschiedlichen Eigenschaften.
- Empirische Standardabweichung und Interquartilsabstand beschreiben die Streuung.
- Histogramm und Boxplot beschreiben die Verteilung eines metrischen Merkmals auf unterschiedliche Weise.
- Gruppenunterschiede und Korrelationen sind zunächst deskriptive Befunde.
- Aus einem statistischen Zusammenhang folgt nicht ohne Weiteres ein kausaler Zusammenhang.
- Kennzahlen aus Zufallsstichproben unterliegen zufälligen Schwankungen.